In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [30]:
df = pd.read_csv("credit_risk_dataset.csv")

In [31]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [32]:
print(df.isnull().sum())

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64


In [50]:
X = df[['person_age','person_income','person_home_ownership','person_emp_length','loan_intent','loan_grade','loan_amnt','loan_int_rate','loan_percent_income','cb_person_default_on_file','cb_person_cred_hist_length']]
y = df[['loan_status']] 

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [52]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(include="object").columns

In [53]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [54]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [55]:
trf = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [60]:
dt = DecisionTreeClassifier(random_state=42,class_weight="balanced")

In [64]:
pipe = Pipeline([
    ("preprocess", trf),
    ("model", dt)
])

In [65]:
param_grid = {
    "model__max_depth": [3, 5, 7, 10],
    "model__min_samples_leaf": [20, 50, 100]
}

In [66]:
gs = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="roc_auc"
)

In [67]:
gs.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length'],
      dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          Sim...st_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         Index(['person_home_ownership', 'loan_intent', 'loan_grade',
       'cb_person_default_on_file'],
      dtype='object'))])),
                                       ('model',
                                        DecisionTreeClassifier(class_weight='balanced',
                                                               random_state=42))]),
             param_grid={'model__max_depth': [3, 5, 7, 10],
                         'model__min_samples_leaf': [20, 50, 100]},
             scoring='roc_auc')

In [68]:
best_model = gs.best_estimator_

In [80]:
print(y_test.iloc[0:1,0:])
print(best_model.predict(X_test.iloc[0:1,0:]))

       loan_status
14668            0
[0]


In [81]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

In [82]:
print("best parameters:", gs.best_params_)

best parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 20}


In [83]:
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.92      0.93      5072
           1       0.73      0.78      0.76      1445

    accuracy                           0.89      6517
   macro avg       0.84      0.85      0.84      6517
weighted avg       0.89      0.89      0.89      6517



In [84]:
print("Test ROC-AUC:", roc_auc_score(y_test, y_prob))

Test ROC-AUC: 0.9224261977011997


In [85]:
# business-defined
threshold = 0.6 
y_pred_custom = (y_prob >= threshold).astype(int)

In [1]:
import joblib
joblib.dump(best_model, "loan_default_model.pkl")

NameError: name 'best_model' is not defined